In [23]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict, Annotated
from langchain_core.messages import BaseMessage, HumanMessage
from langchain_groq import ChatGroq
from langchain_nvidia_ai_endpoints import ChatNVIDIA
from langgraph.graph.message import add_messages
from langgraph.checkpoint.memory import MemorySaver

In [24]:
class ChatState(TypedDict):

    messages: Annotated[list[BaseMessage], add_messages]  # List of messages in the conversation]
    # add_messages is a reducer function 

In [25]:
# llm = ChatNVIDIA(model="nvidia/nemotron-3-super-120b-a12b", temperature=0)
llm = ChatGroq(model="llama-3.3-70b-versatile", temperature=0.2)
def chat_Node(state: ChatState) -> ChatState:

    # take user query from state
    messages = state['messages']

    #send to llm
    response =llm.invoke(messages)

    # response store state
    return {'messages': [response]}


In [26]:
checkpoint = MemorySaver()
graph= StateGraph(ChatState)


# add nodes
graph.add_node('chat_node',chat_Node)

graph.add_edge(START, 'chat_node')
graph.add_edge('chat_node', END)

chatbot =graph.compile(checkpointer=checkpoint)

In [28]:
thread_id = '1'
while True:
    user_message= input('Type here: ')

    print(f'User: {user_message}')

    if user_message.strip().lower() == 'exit':
        break

    config = {'configurable': {'thread_id': thread_id}}

    response = chatbot.invoke({'messages':[HumanMessage(content=user_message)]}, config = config)
    print(f'Bot: {response["messages"][-1].content}')
    # invoke starts from scratch every time, so we need to pass the entire conversation history every time  

User: Hi I'm Tanish
Bot: Hello Tanish, it's nice to meet you. Is there something I can help you with or would you like to chat?
User: WhoAmI?
Bot: It seems like you're asking a philosophical question, Tanish. "Who am I?" is a question that has puzzled humans for centuries, and the answer can be complex and personal.

From a literal perspective, you are Tanish, the person I'm chatting with. But if you're looking for a deeper understanding of yourself, it might involve exploring your values, passions, interests, and experiences.

If you'd like, we can play a game or have a conversation to help you discover more about yourself. Alternatively, I can offer some prompts or questions to help you reflect on your identity. What sounds interesting to you?
User: exit


In [29]:
chatbot.get_state(config=config)

StateSnapshot(values={'messages': [HumanMessage(content="Hi I'm Tanish", additional_kwargs={}, response_metadata={}, id='5794955a-3630-4519-8759-1593b1a9c5f7'), AIMessage(content="Hello Tanish, it's nice to meet you. Is there something I can help you with or would you like to chat?", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 27, 'prompt_tokens': 40, 'total_tokens': 67, 'completion_time': 0.060410978, 'completion_tokens_details': None, 'prompt_time': 0.00998598, 'prompt_tokens_details': None, 'queue_time': 0.16587752, 'total_time': 0.070396958}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_dae98b5ecb', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019ee4cb-c350-7cd2-a0dc-ae1afc7a2291-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 40, 'output_tokens': 27, 'total_tokens': 67}), HumanMessage(content='WhoAmI?', additional_kwargs={}, response_m